In [1]:
import pyspark

In [2]:
pyspark.__file__

'/Users/Durodola/Desktop/PHD/PERSONAL_CODE/Zoomcamp_data/06-batch-processing/spark-4.1.1-bin-hadoop3/python/pyspark/__init__.py'

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/03/08 19:16:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
spark.version

'4.1.1'

In [13]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 19:26:08--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.229, 3.170.186.198, 3.170.186.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.229|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  51.6MB/s    in 1.3s    

2026-03-08 19:26:09 (51.6 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [17]:
df_yellow = spark.read \
            .option("header", "true") \
            .parquet("yellow_tripdata_2025-11.parquet")

In [21]:
df_yellow = df_yellow.repartition(4)

In [22]:
df_yellow.write.parquet('yellow_tripdata_repartitioned')

In [66]:
ls -lh yellow_tripdata_repartitioned/

total 199976
-rw-r--r--  1 Durodola  staff     0B Mar  8 19:42 _SUCCESS
-rw-r--r--  1 Durodola  staff    24M Mar  8 19:42 part-00000-03702fe2-5bc3-4e64-b513-502f1810d11e-c000.snappy.parquet
-rw-r--r--  1 Durodola  staff    24M Mar  8 19:42 part-00001-03702fe2-5bc3-4e64-b513-502f1810d11e-c000.snappy.parquet
-rw-r--r--  1 Durodola  staff    24M Mar  8 19:42 part-00002-03702fe2-5bc3-4e64-b513-502f1810d11e-c000.snappy.parquet
-rw-r--r--  1 Durodola  staff    24M Mar  8 19:42 part-00003-03702fe2-5bc3-4e64-b513-502f1810d11e-c000.snappy.parquet


In [30]:
from pyspark.sql import types

In [31]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True),
    types.StructField("cbd_congestion_fee", types.DoubleType(), True)    
])

In [32]:
df_yellow = spark.read.schema(yellow_schema).parquet('yellow_tripdata_repartitioned/')

In [55]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [34]:
from pyspark.sql import functions as F

In [36]:
df_yellow.withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)) \
            .filter("pickup_date = '2025-11-15'") \
            .count()

167188

In [58]:
df_yellow.filter(
    (F.year("tpep_pickup_datetime") == 2025) &
    (F.month("tpep_pickup_datetime") == 11) &
    (F.year("tpep_dropoff_datetime") == 2025) &
    (F.month("tpep_dropoff_datetime") == 11)
).groupBy(
    F.to_date("tpep_pickup_datetime").alias("date")
).count().orderBy("date").show(50)




+----------+------+
|      date| count|
+----------+------+
|2025-11-01|174131|
|2025-11-02|127121|
|2025-11-03|128930|
|2025-11-04|132337|
|2025-11-05|143322|
|2025-11-06|162801|
|2025-11-07|160097|
|2025-11-08|157380|
|2025-11-09|112883|
|2025-11-10|130797|
|2025-11-11|130440|
|2025-11-12|143467|
|2025-11-13|161384|
|2025-11-14|168404|
|2025-11-15|167188|
|2025-11-16|112285|
|2025-11-17|130073|
|2025-11-18|144360|
|2025-11-19|155175|
|2025-11-20|165689|
|2025-11-21|166236|
|2025-11-22|155769|
|2025-11-23|105501|
|2025-11-24|120710|
|2025-11-25|145896|
|2025-11-26|127775|
|2025-11-27| 91313|
|2025-11-28|105409|
|2025-11-29|119552|
|2025-11-30| 95004|
+----------+------+



In [39]:
df_yellow.select(
    F.min("tpep_pickup_datetime"),
    F.max("tpep_pickup_datetime")
).show()


+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|      2008-12-31 17:04:21|      2025-11-30 17:59:59|
+-------------------------+-------------------------+



In [42]:
df_yellow.groupBy(
    F.year("tpep_pickup_datetime").alias("year"),
    F.month("tpep_pickup_datetime").alias("month")
).count().orderBy("year", "month").show(200)


+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2008|   12|      2|
|2009|    1|      1|
|2025|   10|  39999|
|2025|   11|4141442|
+----+-----+-------+



In [50]:
# What is the length of the longest trip in the dataset in hours?
df_with_duration = df_yellow.withColumn(
    "trip_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
).agg(F.max("trip_hours")) \
.show()

# longest_trip_hours = df_with_duration.agg(F.max("trip_hours")).first()[0]

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [44]:
longest_trip_hours

90.64666666666666

In [46]:
df_yellow \
    .withColumn('duration', (df_yellow.tpep_dropoff_datetime.cast('long') - df_yellow.tpep_pickup_datetime.cast('long')) / 3600) \
    .withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

+-----------+-----------------+
|pickup_date|    max(duration)|
+-----------+-----------------+
| 2025-11-26|90.64666666666666|
| 2025-11-03|76.21388888888889|
| 2025-11-07|69.28861111111111|
| 2025-11-18|67.08055555555555|
| 2025-11-22|63.36833333333333|
+-----------+-----------------+



In [59]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 22:09:17--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.111, 3.170.186.41, 3.170.186.229, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 22:09:17 (5.74 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [63]:
df_zones = spark.read \
            .option("header", "true") \
            .csv('taxi_zone_lookup.csv')

In [64]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [65]:
df_zones.registerTempTable('taxi_zone')

/Users/Durodola/Desktop/PHD/PERSONAL_CODE/Zoomcamp_data/06-batch-processing/spark-4.1.1-bin-hadoop3/python/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [ ]:
from pyspark.sql import functions as F

# Filter to November 2025 only
df_2025_11 = df_yellow.filter(
    (F.year("tpep_pickup_datetime") == 2025) &
    (F.month("tpep_pickup_datetime") == 11)
)

# Count pickups per PULocationID
pickup_counts = (
    df_2025_11.groupBy("PULocationID")
              .count()
)

# Join with zone lookup to get zone names
pickup_with_zones = (
    pickup_counts.join(df_zones, pickup_counts.PULocationID == df_zones.LocationID, "left")
)

# Find the least frequent pickup zone
least_frequent_zone = (
    pickup_with_zones.orderBy("count")
                     .select("Zone", "count")
                     .first()
)

least_frequent_zone


In [75]:
df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID, 'left') \
    .groupBy("Zone") \
    .agg(F.count("*").alias("count")) \
    .orderBy("count") \
    .show()


+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
|       Rikers Island|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|       West Brighton|   14|
|New Dorp/Midland ...|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
